# <b>Cluster Data</b>
The preprocessed matrix is subjected to unsupervised clustering to identify transcriptionally distinct cell populations in the data.

---

## 1. Setup Environment

### 1.1. Import Libraries

In [ ]:
from PIL import Image

import numpy as np
import matplotlib.pyplot as plt

import scanpy as sc

### 1.2. Define File Paths/Directories

In [ ]:
import config

# Preprocessed AnnData file
PREPROCESSED_ANNDATA_FILE = config.PROCESSED_DATA_DIR / "Norman_2019_preprocessed.h5ad"
print("Preprocessed AnnData file:\n", PREPROCESSED_ANNDATA_FILE)

# Clustered AnnData file save directory
CLUSTERED_ANNDATA_FILE = config.PROCESSED_DATA_DIR / "Norman_2019_clustered.h5ad"
print("\nClustered AnnData file will be saved to:\n", CLUSTERED_ANNDATA_FILE)

# Figures save directory
FIGURES_DIR = config.FIGURES_DIR
print("\nFigures will be saved to:\n", FIGURES_DIR)

### 1.3. Define Global Parameters

In [ ]:
SEED = config.SEED

np.random.seed(SEED)
sc.settings.seed = SEED

print(f"\nRandom seed set to: {SEED}")

---

## 2. Load And Validate Data

### 2.1. Load Preprocessed AnnData

In [ ]:
# Read the preprocessed AnnData file
print("\nReading preprocessed AnnData file...")
adata = sc.read_h5ad(PREPROCESSED_ANNDATA_FILE)
print(adata)

### 2.2. Inspect Data

In [ ]:
print("Cell IDs:\n", adata.obs_names)
print("\nGene IDs:\n", adata.var_names)

print("\nGene symbols:\n", adata.var.gene_symbols)

print("\nSample data matrix (first 5 rows and columns):\n", adata.X[:5, :5].toarray())

### 2.3. Validate Data

#### 2.3.1. Verify Normalization and HVG Selection

In [ ]:
# Verify normalization
x_max = adata.X.data.max()
x_min = adata.X.data.min()
assert x_min >= 0, "Negative values found — normalization error"
assert (
    x_max <= np.log1p(1e4) + 0.01
), f"Max value {x_max:.3f} exceeds log1p(10000) — log1p may not have been applied"
print(f"Normalization and log1p verified (value range: {x_min:.3f} - {x_max:.3f})")

# Verify raw counts layer preserved
assert "counts" in adata.layers, "Raw counts layer missing"
assert np.all(
    adata.layers["counts"].data == np.floor(adata.layers["counts"].data)
), "Raw counts layer contains non-integers"
print("Raw counts layer preserved")

# Verify HVGs selected
assert adata.var["highly_variable"].sum() == 3000, "HVG count mismatch"
print("HVGs selected: 3000")

print("\nAll assertions passed.")

#### 2.3.2. Index Genes By Symbol
Index genes by their symbols instead of Ensembl IDs.

In [ ]:
# Check if genes are already indexed by their symbols
are_genes_ens_ids = adata.var_names.str.startswith("ENSG").all()

# Map to gene symbols if currently indexed by Ensembl IDs
if are_genes_ens_ids:
    print("\nGenes are indexed by Ensembl IDs. Mapping to gene symbols....")

    # Check if gene symbols are unique
    if adata.var.gene_symbols.is_unique:
        print("Gene symbols are unique. Setting gene symbols as index....")
        adata.var_names = adata.var['gene_symbols'].values
        print("\nGene symbols set as index successfully.")
    else:
        print("\nGene symbols are NOT unique.")

        # How many duplicates?
        duplicates = adata.var['gene_symbols'][adata.var['gene_symbols'].duplicated(keep=False)]
        print(f"Number of duplicate gene symbol entries: {len(duplicates)}")
        print(duplicates.value_counts().head())

        # Make gene symbols unique by appending a suffix to duplicates
        print("\nMaking gene symbols unique and setting them as index....")
        adata.var_names = adata.var['gene_symbols'].values
        adata.var_names_make_unique()
        print("Gene symbols made unique and set as index successfully.")

# If genes are already indexed by gene symbols, no mapping is needed
else:
    print("\nGenes are already indexed by gene symbols. No mapping needed.")

print("\nGene IDs:\n", adata.var_names)

#### 2.3.4. Verify Uniqueness Of Cell IDs

In [ ]:
are_cell_ids_unique = adata.obs_names.is_unique

if are_cell_ids_unique:
    print("\nCell IDs are unique.")
else:
    print("\nFAILED: Cell IDs are NOT unique.")

---

## 3. Cluster Data

### 3.1. Perform Dimensionality Reduction On HVGs

In [ ]:
# Perform PCA on highly variable genes
print("\nPerforming PCA on highly variable genes...")
sc.tl.pca(adata, n_comps=50, mask_var='highly_variable', random_state=SEED)
print(f"\nPCA complete: {adata.obsm['X_pca'].shape[1]} components computed")

# Plot and save PCA variance ratio
sc.pl.pca_variance_ratio(adata, n_pcs=50, show=False)
plt.savefig(str(config.FIGURES_DIR / "04_pca_variance_ratio.png"), bbox_inches="tight")
plt.show()

### 3.2. Create Neighbor Graph
Since the PCA variance ratio plot has its elbow at `PC6`, a slightly higher number i.e. `n_pcs=10` is used here.

In [ ]:
print("\nComputing neighbors graph...")
sc.pp.neighbors(adata, n_pcs=10, random_state=SEED)
print("Neighbors graph computed.")

### 3.3. Perform Leiden Partitioning

In [ ]:
# Perform Leiden partitioning
print("\nPerforming Leiden partitioning...")
sc.tl.leiden(adata, resolution=0.5, flavor='igraph', n_iterations=2, directed=False, random_state=SEED)
print(f"Leiden partitioning complete.")

print(f"\nNumber of clusters: {adata.obs['leiden'].nunique()}")
print(adata.obs["leiden"].value_counts())

### 3.4. Plot UMAP

#### 3.4.1. Compute UMAP

In [ ]:
sc.tl.umap(adata, random_state=SEED)
print("UMAP computed.")

#### 3.4.2. Plot UMAP Colored By Leiden Clusters

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

leiden_categories = adata.obs["leiden"].cat.categories
colors_leiden = plt.cm.tab10.colors

for i, cluster in enumerate(leiden_categories):
    mask = adata.obs["leiden"] == cluster
    ax.scatter(
        adata.obsm["X_umap"][mask, 0],
        adata.obsm["X_umap"][mask, 1],
        c=[colors_leiden[i % 10]],
        s=0.5,
        alpha=0.5,
        label=cluster,
    )

ax.set_xlabel("UMAP1")
ax.set_ylabel("UMAP2")
ax.set_title("UMAP: Leiden Clusters")
ax.legend(markerscale=5, bbox_to_anchor=(1.05, 1), loc="upper left", title="Cluster")
plt.savefig(str(FIGURES_DIR / "05_umap_leiden.png"), bbox_inches="tight")
plt.show()

#### 3.4.3. Plot UMAP Colored By Perturbation Identity

##### 3.4.3.1. Examine Guide Identity Values

In [ ]:
print(adata.obs["guide_identity"].value_counts().head(20))

##### 3.4.3.2. Create Clean Perturbation Labels

In [ ]:
print("\nCreating clean perturbation labels...")


def simplify_perturbation(label):
    # Take first half of A_B__A_B format, then extract gene names
    first_half = label.split("__")[0]
    parts = first_half.split("_")
    genes = [p for p in parts if not p.startswith("NegCtrl")]
    if len(genes) == 0:
        return "Control"
    return "_".join(genes)


adata.obs["perturbation_clean"] = adata.obs["guide_identity"].apply(
    simplify_perturbation
)
print(adata.obs["perturbation_clean"].value_counts().head(20))

##### 3.4.3.3. Plot UMAP With Perturbations Of Interest

In [ ]:
perturbations_of_interest = ["Control", "CEBPE", "SET", "MAPK1"]

adata.obs["perturbation_focus"] = adata.obs["perturbation_clean"].apply(
    lambda x: x if x in perturbations_of_interest else "Other"
)

fig, ax = plt.subplots(figsize=(8, 6))

# Plot 'Other' cells in light grey background first
other_mask = adata.obs["perturbation_focus"] == "Other"
ax.scatter(
    adata.obsm["X_umap"][other_mask, 0],
    adata.obsm["X_umap"][other_mask, 1],
    c="lightgrey",
    s=0.5,
    alpha=0.3,
    label="Other",
)

# Plot perturbations of interest on top
colors = {
    "Control": "yellow",
    "CEBPE": "blue",
    "SET": "green",
    "MAPK1": "red",
}
for perturbation, color in colors.items():
    mask = adata.obs["perturbation_focus"] == perturbation
    ax.scatter(
        adata.obsm["X_umap"][mask, 0],
        adata.obsm["X_umap"][mask, 1],
        c=color,
        s=1,
        alpha=0.7,
        label=perturbation,
    )

ax.set_xlabel("UMAP1")
ax.set_ylabel("UMAP2")
ax.set_title("UMAP: Selected Perturbations")
ax.legend(markerscale=5, bbox_to_anchor=(1.05, 1), title="Perturbation")
plt.savefig(str(FIGURES_DIR / "06_umap_perturbation.png"), bbox_inches="tight")
plt.show()

### 3.5. Compare Leiden Clusters With Perturbation Identity

In [ ]:
img1 = Image.open(FIGURES_DIR / "05_umap_leiden.png")
img2 = Image.open(FIGURES_DIR / "06_umap_perturbation.png")

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
axes[0].imshow(img1)
axes[0].axis("off")

axes[1].imshow(img2)
axes[1].axis("off")

plt.suptitle("Leiden Clusters vs Perturbation Identity", fontsize=14)
plt.tight_layout()
plt.savefig(str(FIGURES_DIR / "07_umap_comparison.png"), bbox_inches="tight")
plt.show()

Comparing the two UMAP plots, **`CEBPE` (blue in the right plot)** shows the clearest perturbation-driven transcriptional shift. CEBPE-perturbed cells are strongly enriched in the **left protrusion** of the UMAP, which corresponds to **Leiden cluster 6**. This suggests CEBPE overactivation drives K562 cells into a transcriptionally distinct state not observed in control cells.  

**`SET` (green in the right plot)** shows a subtle downward bias within the main body and **`MAPK1` (red in the right plot)** shows a slight leftward shift, but neither achieves the clear spatial separation seen with CEBPE. These perturbations likely have real but more subtle transcriptional effects that are better captured by differential expression analysis than by UMAP visualization alone.

**`CEBPE`** will therefore be the **primary focus** of downstream differential expression and pathway enrichment analysis.

## 4. Perform Final Sanity Checks

In [ ]:
print("=== POST-CLUSTERING SANITY CHECK ===\n")

# Verify PCA was computed on HVGs
assert "X_pca" in adata.obsm, "PCA not found"
assert adata.obsm["X_pca"].shape[1] == 50, "PCA component count mismatch"
print(
    f"PCA computed: {adata.obsm['X_pca'].shape[1]} components on {adata.var['highly_variable'].sum()} HVGs"
)

# Verify neighbors graph
assert "neighbors" in adata.uns, "Neighbors graph not found"
assert adata.uns["neighbors"]["params"]["n_pcs"] == 10, "n_pcs mismatch"
print(f"Neighbors graph computed: n_pcs=10")

# Verify leiden clustering
assert "leiden" in adata.obs.columns, "Leiden clusters not found"
n_clusters = adata.obs["leiden"].nunique()
assert n_clusters == 7, f"Expected 7 clusters, got {n_clusters}"
print(f"Leiden clustering: {n_clusters} clusters at resolution=0.5")

# Verify UMAP
assert "X_umap" in adata.obsm, "UMAP not found"
assert adata.obsm["X_umap"].shape[1] == 2, "UMAP should have 2 dimensions"
print(f"UMAP computed: {adata.obsm['X_umap'].shape[1]} dimensions")

# Verify perturbation columns exist
assert "perturbation_clean" in adata.obs.columns, "perturbation_clean column missing"
assert "perturbation_focus" in adata.obs.columns, "perturbation_focus column missing"
print(f"Perturbation columns created")

# Verify selected perturbations exist
for p in ["Control", "CEBPE", "SET", "MAPK1"]:
    assert (
        p in adata.obs["perturbation_clean"].values
    ), f"{p} not found in perturbation_clean"
print(f"All selected perturbations present: Control, CEBPE, SET, MAPK1")

print("\nAll assertions passed.")

---

## 5. Save Clustered AnnData

In [ ]:
adata.write_h5ad(CLUSTERED_ANNDATA_FILE)
print(f"Saved: {CLUSTERED_ANNDATA_FILE}")

---

## 6. Summary

### 6.1. Data Validation
- Gene symbols confirmed as index (no Ensembl IDs)
- Cell barcodes confirmed unique
- Normalization and HVGs verified

### 6.2. Steps Executed
1. **PCA** — dimensionality reduction to 50 components using 3,000 HVGs; elbow identified at PC6, informing n_pcs=10 for neighbors graph
2. **Neighbors graph** — computed using n_pcs=10, n_neighbors=15 (default)
3. **Leiden partitioning** — resolution=0.5, flavor=igraph; 7 clusters identified
4. **UMAP** — 2D embedding computed for visualization
5. **Perturbation visualization** — iteratively explored perturbations; CEBPE, SET, and MAPK1 selected as perturbations of interest based on visible UMAP shifts relative to control

### 6.3. Key Findings
- **CEBPE** shows the strongest transcriptional shift — enriched in the left protrusion of the UMAP, partially corresponding to Leiden cluster 6
- **SET** shows a subtle downward bias within the main body
- **MAPK1** shows a slight leftward shift within the main body
- Neither SET nor MAPK1 achieves the spatial separation seen with CEBPE

### 6.4. Parameters
- PCA: n_comps=50, use_highly_variable=True
- Neighbors: n_pcs=10, n_neighbors=15
- Leiden: resolution=0.5, flavor=igraph, n_iterations=2
- UMAP: default parameters